In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install py_vncorenlp underthesea pyserini faiss-cpu

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 9.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.2/412.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os

def install_java():
  !apt update -qq
  !apt-get install -y openjdk-21-jdk-headless -qq > /dev/null
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
  !java -version

install_java()

3 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "21.0.9" 2025-10-21
OpenJDK Runtime Environment (build 21.0.9+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.9+10-Ubuntu-122.04, mixed mode, sharing)


In [ ]:
!git clone https://github.com/tommachilez/virag4fc.git
%cd /content/virag4fc/

Cloning into 'fact-checking-data-framework-vn'...
remote: Enumerating objects: 309, done.
remote: Counting objects: 100% (309/309), done.
remote: Compressing objects: 100% (199/199), done.
remote: Total 309 (delta 182), reused 232 (delta 108), pack-reused 0 (from 0)
Receiving objects: 100% (309/309), 5.31 MiB | 15.68 MiB/s, done.
Resolving deltas: 100% (182/182), done.
/content/fact-checking-data-framework-vn


In [5]:
VNCORE_MODEL_PATH = "/content/drive/MyDrive/KLTN_Project/Models/vncorenlp_models"
csv_path = "/content/drive/MyDrive/KLTN_Project/Datasets/vifc.csv"
jsonl_path = "/content/drive/MyDrive/KLTN_Project/Datasets/queries_triples.jsonl"
stopwords_path = "/content/drive/MyDrive/KLTN_Project/Datasets/stopwords-vi.txt"
output_path = "/content/drive/MyDrive/KLTN_Project/Datasets/training_triples.jsonl"
mining_dir = "/content/drive/MyDrive/KLTN_Project/Datasets/vn_mining"

In [6]:
# Step 1: Preprocessing (Uses PyVnCoreNLP)
!python -m src.scripts.hard_negative_mining.preprocess \
    --doc_csv {csv_path} \
    --query_jsonl {jsonl_path} \
    --vncorenlp_path {VNCORE_MODEL_PATH} \
    --stopwords_path {stopwords_path} \
    --output_dir {mining_dir} \
    --doc_col "document" \
    --id_col "id"

2025-12-18 09:10:25 INFO  WordSegmenter:24 - Loading Word Segmentation model
>>> Processing Documents (Deduplicating)...
Deduping Docs: 34811it [00:50, 694.59it/s] 
>>> Saving deduplicated document map (3030 unique docs)...
>>> Processing Queries...
Queries: 2692it [00:06, 384.70it/s]
>>> Preprocessing complete. Files saved to /content/drive/MyDrive/KLTN_Project/Datasets/vn_mining


In [7]:
# Step 2: Mining (Uses Pyserini)
# Note: This is a separate process execution, so a new JVM is started.
!python -m src.scripts.hard_negative_mining.pyserini_mining \
    --preprocessed_dir {mining_dir} \
    --original_doc_csv {csv_path} \
    --output_jsonl {output_path} \
    --top_k 10 \
    --doc_col "document" \
    --id_col "id"

2025-12-18 09:11:37.739761: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-18 09:11:37.745448: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-18 09:11:37.758843: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766049097.783133    2006 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766049097.789806    2006 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766049097.807851    2006 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [8]:
import json
import os

class DatasetViewer:
    def __init__(self, corpus_path: str, queries_path: str, triples_path: str):
        """
        Initialize with paths to the three key files.
        """
        self.paths = {
            "corpus": corpus_path,
            "queries": queries_path,
            "triples": triples_path
        }

    def _view_file(self, key: str, n: int):
        path = self.paths.get(key)
        title = key.replace('_', ' ').title()

        print(f"\n{'='*20} VIEWING: {title} (First {n} lines) {'='*20}")

        if not path or not os.path.exists(path):
            print(f"❌ File not found at: {path}")
            return

        try:
            with open(path, 'r', encoding='utf-8') as f:
                for i in range(n):
                    line = f.readline()
                    if not line:
                        print(f"--- End of file reached at line {i} ---")
                        break

                    try:
                        # Parse JSON to pretty-print it
                        data = json.loads(line.strip())
                        print(f"🔹 [Line {i+1}]")
                        print(json.dumps(data, indent=2, ensure_ascii=False))
                    except json.JSONDecodeError:
                        print(f"⚠️ [Line {i+1}] (Invalid JSON)")
                        print(line.strip())
        except Exception as e:
            print(f"Error reading file: {e}")

    def view_corpus(self, n: int = 5):
        """View the pretokenized corpus file."""
        self._view_file("corpus", n)

    def view_queries(self, n: int = 5):
        """View the pretokenized queries file."""
        self._view_file("queries", n)

    def view_triples(self, n: int = 5):
        """View the final output training triples."""
        self._view_file("triples", n)

    def view_all(self, n: int = 3):
        """View first n lines of all files sequentially."""
        self.view_corpus(n)
        self.view_queries(n)
        self.view_triples(n)

In [11]:
viewer = DatasetViewer(
    f"{mining_dir}/corpus_pretokenized.jsonl",
    f"{mining_dir}/queries_pretokenized.jsonl",
    output_path,
)

In [12]:
viewer.view_all(5)


==================== VIEWING: Corpus (First 5 lines) ====================
🔹 [Line 1]
{
  "id": "55f5ff9002d5ed369d65afcfb89aae97",
  "contents": "chinhphu.vn là mong_muốn gửi_gắm của phó thủ_tướng trần hồng hà đến truyền_hình tại lễ bế_mạc liên_hoan truyền_hình toàn_quốc thứ 41 tối 18/3 tại tp hải_phòng phó thủ_tướng trần hồng hà tác_phẩm truyền_hình vun_đắp làm_giàu văn_hoá việt_nam tiên_tiến đậm_đà bản_sắc dân_tộc góp_phần tạo_dựng môi_trường văn_hoá lành_mạnh và xây_dựng con_người việt_nam nhân_cách trách_nhiệm hội_nhập ảnh vgp minh khôi tham_dự lễ bế_mạc bí_thư trung_ương đảng trưởng ban tuyên_giáo trung_ương nguyễn_trọng nghĩa lãnh_đạo ngành trung_ương địa_phương đại_diện đài_truyền_hình đơn_vị sản_xuất chương_trình truyền_hình đông_đảo cán_bộ phóng_viên biên_tập_viên nghệ_sĩ diễn_viên hoạt_động trong lĩnh_vực truyền_hình … thay_mặt chính_phủ thủ_tướng chính_phủ phó thủ_tướng trần hồng hà chúc_mừng đài_truyền_hình việt_nam đài_truyền_hình tỉnh thành_phố trên nước đơn_vị sản_xuất 